# Anchor-based Li site extraction for LPSC (48h + 48h' + 16e)

This notebook is a thin driver that orchestrates the `lpsc_sites` package
(see `../lpsc_sites/` for the implementation and `../tests/` for the
property tests). The scientific motivation and method are reproduced below;
the code cells are intentionally short.

**Idea.** The framework sublattice (P, S, O, Cl) is static on the AIMD
timescale (MSD ≈ 0), so the Li Wyckoff site geometry is fixed relative to
the framework. Rather than detecting peaks in the Li density with watershed
and classifying them post-hoc — which fails when peaks merge or when noise
over-segments them — we place candidate sites at the expected Wyckoff
positions, rigidly align to the AIMD frame via a P anchor, and refine each
candidate to the local density maximum with an iterative empirical-canonical
learning loop.

**Workflow.**

1. Load the reference CIF; preserve Wyckoff labels through supercell replication.
2. Load the AIMD trajectory; compute the time-averaged framework (P, S, O, Cl).
3. Recover the rigid shift from the P anchor.
4. Classify each 4a / 4d cage as *normal* or *swapped*.
5. Generate Li candidates from the expanded ideal positions.
6. Build the Li density volume and refine each candidate.
7. Iteratively learn the empirical Wyckoff canonical; run a tight Pass 2.
8. Flag candidates that collapse onto the same density peak.
9. Export the labelled CIF and a JSON metadata sidecar.

In [ ]:
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from pymatgen.io.cif import CifParser
from gemdat import Trajectory

from lpsc_sites import (
    DensityVolume,
    build_labeled_structure, classify_cages, export_metadata,
    find_duplicate_conflicts, generate_candidates, ideal_fracs_by_label,
    learn_empirical_canonicals, min_pbc_distances_A, pbc_mean_frac, pbc_mic,
    recover_rigid_shift, refine_candidates, replicate_preserving_labels,
    reset_candidates_to_cif,
)

warnings.filterwarnings('ignore')

In [ ]:
# --- Input / output paths ---
CACHE         = '../data/reduced.cache'
REFERENCE_CIF = '../data/LPSC_all.cif'
OUTPUT_CIF    = '../data/sites_anchor.cif'

# --- Trajectory sampling ---
NR_EQUILIBRATION_STEPS =  0   # cache already trimmed to post-equilibration frames
SUPERCELL: tuple[int, int, int] = (2, 1, 1)

# --- Density volume ---
RESOLUTION_A          = 0.16
REFINE_RADIUS_A       = 0.8
REFINE_RADIUS_PASS2_A = 0.3
INTEGRATE_RADIUS_A    = 0.5
DUPLICATE_MERGE_A     = 0.3

# --- Iterative canonical learning ---
MAX_ITER               = 5
CONVERGE_A             = 0.05
RUNAWAY_A              = 0.6
CLEAN_DISP_A           = 0.55
MIN_CLEAN_FOR_LEARNING = 10

MAX_RIGID_SHIFT_RESIDUAL_A = 0.5

# --- Labels from the reference CIF ---
LI_LABELS: tuple[str, ...] = ('48h', '48h2', '16e')
P_LABEL, CL_LABEL, S_4D_LABEL = 'P1', 'Cl4', 'S3'
HALOGENS, CHALCOGENS = frozenset({'Cl'}), frozenset({'S', 'O'})

SAVE_OUTPUTS = True

## 1. Reference CIF and label-preserving supercell

In [ ]:
parser = CifParser(REFERENCE_CIF)
ref_struct = parser.parse_structures(primitive=False, symmetrized=False)[0]
super_ref  = replicate_preserving_labels(ref_struct, SUPERCELL)

print(f"Reference: {ref_struct.formula}, a = {ref_struct.lattice.a:.4f} Å (F-43m)")
print(f"Supercell ({SUPERCELL}): {super_ref.formula}, N = {len(super_ref)}")
print("\nLabel counts in supercell:")
for lab, n in sorted(Counter(s.label for s in super_ref.sites).items()):
    print(f"  {lab:>6}: {n}")

In [ ]:
ideal = {
    'P':     ideal_fracs_by_label(super_ref, P_LABEL),
    'Cl_4a': ideal_fracs_by_label(super_ref, CL_LABEL),
    'S_4d':  ideal_fracs_by_label(super_ref, S_4D_LABEL),
    **{lab: ideal_fracs_by_label(super_ref, lab) for lab in LI_LABELS},
}
print("Ideal (unshifted) arrays:")
for k, v in ideal.items():
    print(f"  {k:>8}: {len(v):>3}")

## 2. Trajectory and time-averaged framework

The time-averaged framework absorbs thermal distortions induced by disorder
and oxygen substitution; this is what distinguishes the method from a pure
ideal-CIF overlay.

In [ ]:
trajectory = Trajectory.from_vasprun(VASPRUN, cache=CACHE)
trajectory = trajectory[NR_EQUILIBRATION_STEPS:]
print(trajectory)

aimd_lat = trajectory.get_lattice()
nominal_a = super_ref.lattice.a
print(f"\nAIMD lattice a = {aimd_lat.a:.3f} Å "
      f"(reference × supercell: {nominal_a:.3f} Å, "
      f"diff = {(aimd_lat.a - nominal_a) / nominal_a * 100:+.2f}%)")

In [ ]:
species_per_atom = np.array([s.symbol for s in trajectory.species])
positions_all    = trajectory.positions

framework = {
    sym: pbc_mean_frac(positions_all[:, species_per_atom == sym, :])
    for sym in ('P', 'Cl', 'S', 'O')
}
print("Framework atoms (time-averaged):")
for sym, arr in framework.items():
    print(f"  {sym}: {len(arr)}")

## 3. Rigid shift from the P anchor

P occupies Wyckoff 4b and is unaffected by the 4a/4d swap, so it gives a
clean rigid-translation estimate between CIF and AIMD frames.

In [ ]:
shift_frac = recover_rigid_shift(ideal['P'], framework['P'])
residuals  = min_pbc_distances_A(framework['P'], ideal['P'], shift_frac, aimd_lat)

print(f"Rigid shift (frac): {np.round(shift_frac, 4)}")
print(f"Residual P displacement: mean = {residuals.mean():.3f} Å, max = {residuals.max():.3f} Å")
if residuals.max() > MAX_RIGID_SHIFT_RESIDUAL_A:
    print(f"⚠ Residual > {MAX_RIGID_SHIFT_RESIDUAL_A} Å — rigid shift may not capture "
          "the cell distortion; consider per-cage alignment.")

## 4. Cage classification (4a / 4d, normal vs swapped)

In [ ]:
cage_4a_status = classify_cages(ideal['Cl_4a'], framework, shift_frac, aimd_lat, HALOGENS)
cage_4d_status = classify_cages(ideal['S_4d'],  framework, shift_frac, aimd_lat, CHALCOGENS)

for name, status in [('4a', cage_4a_status), ('4d', cage_4d_status)]:
    n_norm = sum(c.status == 'normal' for c in status)
    n_swap = len(status) - n_norm
    kind   = 'Cl' if name == '4a' else 'S/O'
    print(f"{name}: {n_norm}/{len(status)} normal ({kind})  {n_swap} swapped")

n_swap_4a = sum(c.status == 'swapped' for c in cage_4a_status)
n_swap_4d = sum(c.status == 'swapped' for c in cage_4d_status)
if n_swap_4a != n_swap_4d:
    print(f"⚠ 4a swaps ({n_swap_4a}) ≠ 4d swaps ({n_swap_4d}) — expected equal "
          "for 4a↔4d pair-swaps.")

## 5. Li candidates (labels attached by construction)

In [ ]:
candidates = generate_candidates(
    ideal_by_label=ideal,
    li_labels=LI_LABELS,
    cage_status=cage_4d_status,
    cage_fracs=ideal['S_4d'],
    shift_frac=shift_frac,
    cage_type='S_4d',
)

print(f"Total candidates: {len(candidates)}\n")
print("Cage status distribution (cage centred on ideal 4d):")
for lab in LI_LABELS:
    n_norm = sum(1 for c in candidates if c.wyckoff == lab and c.cage_status == 'normal')
    n_swap = sum(1 for c in candidates if c.wyckoff == lab and c.cage_status == 'swapped')
    print(f"  {lab:>6}: {n_norm:>3} normal (S at 4d) | {n_swap:>3} swapped (Cl at 4d)")

## 6. Li density volume

In [ ]:
volume = DensityVolume.from_gemdat(
    trajectory.filter('Li').to_volume(resolution=RESOLUTION_A),
    aimd_lat,
)
print(f"Density grid: {tuple(volume.shape)}, voxel size = {np.round(volume.voxel_size_A, 3)} Å")
print(f"Density max = {volume.density.max():.2f}, mean = {volume.density.mean():.2f}")

## 7. Pass 1 — wide-ROI refinement (diagnostic)

A generous ROI bridges the initial mismatch between CIF ideal positions and
AIMD density maxima. *Wall-huggers* — candidates whose refined position sits
at the ROI boundary — motivate the iterative canonical learning that follows.

In [ ]:
refine_candidates(candidates, volume, REFINE_RADIUS_A)

print("Pass 1 refinement displacement (Å):")
for lab in LI_LABELS:
    d = [c.refinement_disp_A for c in candidates if c.wyckoff == lab]
    wall_huggers = sum(x > 0.95 * REFINE_RADIUS_A for x in d)
    print(f"  {lab:>6}: mean={np.mean(d):.3f}  max={max(d):.3f}  "
          f"n={len(d)}  wall-huggers={wall_huggers}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.linspace(0, REFINE_RADIUS_A * 1.05, 30)
colors = dict(zip(LI_LABELS, plt.cm.tab10(np.linspace(0, 1, len(LI_LABELS)))))
for lab in LI_LABELS:
    d = [c.refinement_disp_A for c in candidates if c.wyckoff == lab]
    ax.hist(d, bins=bins, alpha=0.6, color=colors[lab],
            edgecolor='black', linewidth=0.5,
            label=f"{lab} (n={len(d)}, mean={np.mean(d):.2f} Å)")
ax.axvline(REFINE_RADIUS_A, color='red', linestyle='--', label=f'ROI = {REFINE_RADIUS_A} Å')
ax.set(xlabel='Refinement displacement (Å)', ylabel='Count',
       title='Pass 1 refinement — displacement from CIF ideal')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 8. Iterative empirical-canonical learning

For each Wyckoff orbit, we refine the "clean" candidates, fold them onto the
current canonical under F-43m symmetry, take the circular mean, project onto
the Wyckoff free-parameter surface, re-seed the candidates, and iterate.

In [ ]:
reset_candidates_to_cif(candidates, ideal, shift_frac, LI_LABELS)
canonical_cif, empirical_canonical = learn_empirical_canonicals(
    candidates=candidates,
    volume=volume,
    ideal=ideal,
    shift_frac=shift_frac,
    primitive_lattice=ref_struct.lattice,
    supercell=SUPERCELL,
    li_labels=LI_LABELS,
    refine_radius_A=REFINE_RADIUS_A,
    max_iter=MAX_ITER,
    clean_disp_A=CLEAN_DISP_A,
    converge_A=CONVERGE_A,
    runaway_A=RUNAWAY_A,
    min_clean_for_learning=MIN_CLEAN_FOR_LEARNING,
)

print("\nFinal empirical canonicals:")
for lab in LI_LABELS:
    d_A = np.linalg.norm(pbc_mic(empirical_canonical[lab] - canonical_cif[lab])) * ref_struct.lattice.a
    print(f"  {lab:>5}: {np.round(empirical_canonical[lab], 4)}   (Δ from CIF = {d_A:.3f} Å)")

## 9. Pass 2 — tight refinement from the learned canonical

In [ ]:
refine_candidates(candidates, volume, REFINE_RADIUS_PASS2_A, INTEGRATE_RADIUS_A)

print(f"Pass 2 refinement (ROI = {REFINE_RADIUS_PASS2_A} Å):")
for lab in LI_LABELS:
    d = [c.refinement_disp_A for c in candidates if c.wyckoff == lab]
    print(f"  {lab:>6}: mean={np.mean(d):.3f}  max={max(d):.3f}  n={len(d)}")

## 10. Duplicate detection

In [ ]:
conflict_pairs = find_duplicate_conflicts(candidates, aimd_lat, DUPLICATE_MERGE_A)
n_conflict = sum(1 for c in candidates if c.conflicts_with)
print(f"Conflicts (refined distance < {DUPLICATE_MERGE_A} Å): {n_conflict} candidates")

if conflict_pairs:
    pair_counter = Counter(
        tuple(sorted([candidates[i].full_label, candidates[j].full_label]))
        for i, j in conflict_pairs
    )
    print("\nConflict pairs by label:")
    for pair, n in pair_counter.most_common():
        print(f"  {pair[0]:>6} ↔ {pair[1]:<6}: {n}")

## 11. Diagnostic plots — final displacements and integrated density

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

bins_disp = np.linspace(0, REFINE_RADIUS_PASS2_A * 1.05, 30)
for lab in LI_LABELS:
    d = [c.refinement_disp_A for c in candidates if c.full_label == lab]
    axes[0].hist(d, bins=bins_disp, alpha=0.6, color=colors[lab], label=f"{lab} (n={len(d)})")
axes[0].axvline(REFINE_RADIUS_PASS2_A, color='red', linestyle='--',
                label=f'ROI = {REFINE_RADIUS_PASS2_A} Å')
axes[0].set(xlabel='Refinement displacement (Å)', ylabel='Count',
            title='Candidate → local-max distance (Pass 2)')
axes[0].legend(fontsize=8)

max_int = max(c.integrated_density for c in candidates)
bins_int = np.linspace(0, max_int * 1.05, 30)
for lab in LI_LABELS:
    d = [c.integrated_density for c in candidates if c.full_label == lab]
    axes[1].hist(d, bins=bins_int, alpha=0.6, color=colors[lab], label=f"{lab} (n={len(d)})")
axes[1].set(xlabel=f'Integrated density (sphere r = {INTEGRATE_RADIUS_A} Å)', ylabel='Count',
            title='Integrated density per candidate')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 12. Labelled structure + CIF + metadata export

In [ ]:
labeled = build_labeled_structure(candidates, aimd_lat)

if SAVE_OUTPUTS:
    labeled.to(filename=OUTPUT_CIF)
    print(f"Wrote {len(labeled)} labelled sites → {OUTPUT_CIF}")

    meta_path = export_metadata(
        Path(OUTPUT_CIF).with_suffix('.json'),
        trajectory_path=VASPRUN,
        reference_cif=REFERENCE_CIF,
        output_cif=OUTPUT_CIF,
        supercell=SUPERCELL,
        n_equilibration_steps=NR_EQUILIBRATION_STEPS,
        aimd_lattice=aimd_lat,
        shift_frac=shift_frac,
        params={
            'resolution_A':          RESOLUTION_A,
            'refine_radius_A':       REFINE_RADIUS_A,
            'refine_radius_pass2_A': REFINE_RADIUS_PASS2_A,
            'integrate_radius_A':    INTEGRATE_RADIUS_A,
            'duplicate_merge_A':     DUPLICATE_MERGE_A,
        },
        cage_4a_status=cage_4a_status,
        cage_4d_status=cage_4d_status,
        candidates=candidates,
        labeled=labeled,
    )
    print(f"Wrote metadata      → {meta_path}")
else:
    print("SAVE_OUTPUTS is False — CIF and metadata not written.")

## 13. Final site inventory

In [ ]:
by_label   = Counter(c.wyckoff for c in candidates)
by_normal  = Counter(c.wyckoff for c in candidates if c.cage_status == 'normal')
by_swapped = Counter(c.wyckoff for c in candidates if c.cage_status == 'swapped')
display    = {'48h': '48h', '48h2': "48h'", '16e': '16e'}

print(f"{'Wyckoff':<10}{'total':>10}{'normal':>10}{'swapped':>10}")
print('-' * 40)
for lab in LI_LABELS:
    print(f"{display[lab]:<10}{by_label[lab]:>10}{by_normal[lab]:>10}{by_swapped[lab]:>10}")
print('-' * 40)
print(f"{'TOTAL':<10}{sum(by_label.values()):>10}")

n_conflict = sum(1 for c in candidates if c.conflicts_with)
print(f"\nConflicts (refined distance < {DUPLICATE_MERGE_A} Å): {n_conflict}/{len(candidates)}")

## 14. 3D visual check — density + labelled sites

`DensityVolume.plot_3d` delegates to the underlying `gemdat.Volume`, so any
keyword argument gemdat accepts works here.

In [ ]:
fig = volume.plot_3d(structure=labeled)
fig.show()